# Modelo GLM

Sirve para ver que featurs son mejores

In [1]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [2]:
with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

In [3]:
df_data.head()

,started_at,ride_id,ended_at,member_casual,year,month,day,time_hms_ms,event,start_station_id,end_station_id,temperature,wind_speed,precipitation,relative_humidity,snow_depth,rideable_type_classic_bike,rideable_type_docked_bike,rideable_type_electric_bike,member_casual_bool,day_type_Holiday,day_type_Normal,day_type_Weekend,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,start_station_idx,end_station_idx,hour_float,hour_sin,hour_cos,month_sin,month_cos,duration_min
0,2022-01-04 04:38:28,12E6E4CC93D5FDBB,2022-01-04 04:51:48,casual,2022,1,4,0 days 04:38:28,False,8fd03909-317e-4b6c-b8e9-f4c28b4c90b1,bbb67562-1f82-4dfa-a532-231ddc35444e,-5.47,3.100000,0.0,74.85,0.0,True,False,False,True,False,True,False,-2.348318,-0.573362,0.916001,-0.099704,-0.056148,1281,1896,4.641111,0.937383,0.348299,0.5,0.866025,13.333333
1,2022-01-04 11:26:49,03A4B2FF57D83F87,2022-01-04 11:35:30,casual,2022,1,4,0 days 11:26:49,False,a3b2905a-a135-11e9-9cda-0a87ae2ba916,a3ad09b2-a135-11e9-9cda-0a87ae2ba916,-5.00,4.600000,0.0,80.00,0.0,True,False,False,True,False,True,False,-2.299739,0.115036,1.211601,-0.099704,-0.056148,1819,1645,11.446944,0.144284,-0.989536,0.5,0.866025,8.683333
2,2022-01-04 13:13:55,656B583500F94041,2022-01-04 13:52:27,member,2022,1,4,0 days 13:13:55,False,2a80f8d9-85dc-4c36-ba07-86d775ffd1ee,2a80f8d9-85dc-4c36-ba07-86d775ffd1ee,-5.00,4.841667,0.0,82.90,0.0,True,False,False,False,False,True,False,-2.299739,0.225945,1.378055,-0.099704,-0.056148,1250,1250,13.231944,-0.316960,-0.948439,0.5,0.866025,38.533333
3,2022-01-12 15:29:54,E1E90907D0DAAD57,2022-01-12 15:36:05,casual,2022,1,12,0 days 15:29:54,False,bbb67562-1f82-4dfa-a532-231ddc35444e,ff21c158-de98-46c6-9ef7-7082bdb7c634,0.75,2.850000,0.0,76.25,0.0,False,False,True,True,False,True,False,-1.705422,-0.688095,0.996358,-0.099704,-0.056148,1896,1911,15.498333,-0.793088,-0.609108,0.5,0.866025,6.183333
4,2022-01-16 08:28:04,8CDE4F5C9F79729C,2022-01-16 08:58:12,casual,2022,1,16,0 days 08:28:04,False,b09832de-1c8e-4b92-bd72-bd98dc80b92c,a3ad0ecb-a135-11e9-9cda-0a87ae2ba916,-9.83,1.885000,0.0,74.30,0.0,True,False,False,True,False,False,True,-2.798966,-1.130964,0.884432,-0.099704,-0.056148,1893,1646,8.467778,0.798460,-0.602047,0.5,0.866025,30.133333


In [4]:
df_model = df_data.drop(columns=[
    "ride_id",
    "ended_at",
    "time_hms_ms",
    "member_casual",
    "start_station_id",
    "end_station_id",
    "month",
    "day",
    "temperature",
    "wind_speed",
    "precipitation",
    "relative_humidity",
    "snow_depth",
    "hour_float"
])

In [5]:
df_model.head()

,started_at,year,event,rideable_type_classic_bike,rideable_type_docked_bike,rideable_type_electric_bike,member_casual_bool,day_type_Holiday,day_type_Normal,day_type_Weekend,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,start_station_idx,end_station_idx,hour_sin,hour_cos,month_sin,month_cos,duration_min
0,2022-01-04 04:38:28,2022,False,True,False,False,True,False,True,False,-2.348318,-0.573362,0.916001,-0.099704,-0.056148,1281,1896,0.937383,0.348299,0.5,0.866025,13.333333
1,2022-01-04 11:26:49,2022,False,True,False,False,True,False,True,False,-2.299739,0.115036,1.211601,-0.099704,-0.056148,1819,1645,0.144284,-0.989536,0.5,0.866025,8.683333
2,2022-01-04 13:13:55,2022,False,True,False,False,False,False,True,False,-2.299739,0.225945,1.378055,-0.099704,-0.056148,1250,1250,-0.316960,-0.948439,0.5,0.866025,38.533333
3,2022-01-12 15:29:54,2022,False,False,False,True,True,False,True,False,-1.705422,-0.688095,0.996358,-0.099704,-0.056148,1896,1911,-0.793088,-0.609108,0.5,0.866025,6.183333
4,2022-01-16 08:28:04,2022,False,True,False,False,True,False,False,True,-2.798966,-1.130964,0.884432,-0.099704,-0.056148,1893,1646,0.798460,-0.602047,0.5,0.866025,30.133333


In [6]:
# Se redonde al minuto mas cercano
df_data["started_minute"] = df_data["started_at"].dt.round("min")

In [7]:
df_model.head()

,started_at,year,event,rideable_type_classic_bike,rideable_type_docked_bike,rideable_type_electric_bike,member_casual_bool,day_type_Holiday,day_type_Normal,day_type_Weekend,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,start_station_idx,end_station_idx,hour_sin,hour_cos,month_sin,month_cos,duration_min
0,2022-01-04 04:38:28,2022,False,True,False,False,True,False,True,False,-2.348318,-0.573362,0.916001,-0.099704,-0.056148,1281,1896,0.937383,0.348299,0.5,0.866025,13.333333
1,2022-01-04 11:26:49,2022,False,True,False,False,True,False,True,False,-2.299739,0.115036,1.211601,-0.099704,-0.056148,1819,1645,0.144284,-0.989536,0.5,0.866025,8.683333
2,2022-01-04 13:13:55,2022,False,True,False,False,False,False,True,False,-2.299739,0.225945,1.378055,-0.099704,-0.056148,1250,1250,-0.316960,-0.948439,0.5,0.866025,38.533333
3,2022-01-12 15:29:54,2022,False,False,False,True,True,False,True,False,-1.705422,-0.688095,0.996358,-0.099704,-0.056148,1896,1911,-0.793088,-0.609108,0.5,0.866025,6.183333
4,2022-01-16 08:28:04,2022,False,True,False,False,True,False,False,True,-2.798966,-1.130964,0.884432,-0.099704,-0.056148,1893,1646,0.798460,-0.602047,0.5,0.866025,30.133333


In [8]:
df_agg = df_data.groupby([
    "start_station_idx",
    "end_station_idx",
    "started_minute"
]).agg(
    n_viajes=("ride_id", "count"),
    year=("year", "first"),
    temp_std=("temp_std", "first"),
    wind_std=("wind_std", "first"),
    rel_humidity_std=("rel_humidity_std", "first"),
    precipitation_std=("precipitation_std", "first"),
    snow_depth_std=("snow_depth_std", "first"),
    hour_sin=("hour_sin", "first"),
    hour_cos=("hour_cos", "first"),
    month_sin=("month_sin", "first"),
    month_cos=("month_cos", "first"),
    event=("event", "any"),  # True si al menos un dato es true
    normal_day=("day_type_Normal", "any"),  # True si al menos un dato es true
    weekend_day=("day_type_Weekend", "any"),  # True si al menos un dato es true
    holiday_day=("day_type_Holiday", "any"),  # True si al menos un dato es true
    #member_casual=("member_casual_bool", "any"),  # True si al menos un dato es true
    #classic_bike=("rideable_type_classic_bike", "any"),  # True si al menos un dato es true
    #docked_bike=("rideable_type_docked_bike", "any"),  # True si al menos un dato es true
    #electric_bike=("rideable_type_electric_bike", "any"),  # True si al menos un dato es true
    #duration_min_mean=("duration_min", "mean"),
).reset_index()

In [9]:
df_agg.head()

,start_station_idx,end_station_idx,started_minute,n_viajes,year,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_sin,hour_cos,month_sin,month_cos,event,normal_day,weekend_day,holiday_day
0,0,0,2022-02-11 13:38:00,1,2022,-1.379839,1.021427,1.326397,-0.099704,-0.056148,-0.416215,-0.909266,0.866025,5.000000e-01,False,False,False,True
1,0,0,2022-03-05 15:03:00,1,2022,-0.612834,-0.276355,-0.807078,-0.099704,-0.056148,-0.717569,-0.696487,1.000000,6.123234e-17,False,False,True,False
2,0,0,2022-03-06 10:55:00,1,2022,-1.156238,4.113864,0.687365,-0.099704,-0.056148,0.281155,-0.959662,1.000000,6.123234e-17,False,False,True,False
3,0,0,2022-03-20 17:06:00,1,2022,-0.424108,-0.883141,-1.019270,-0.099704,-0.056148,-0.972183,-0.234223,1.000000,6.123234e-17,False,False,True,False
4,0,0,2022-03-20 17:11:00,1,2022,-0.410327,-0.825775,-1.038403,-0.099704,-0.056148,-0.976781,-0.214238,1.000000,6.123234e-17,False,False,True,False


In [10]:
df_agg.describe()

,start_station_idx,end_station_idx,started_minute,n_viajes,year,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_sin,hour_cos,month_sin,month_cos
count,9.073410e+06,9.073410e+06,9073410,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06,9.073410e+06
mean,1.451035e+03,1.451958e+03,2023-03-15 06:23:51.479779072,1.048204e+00,2.022696e+03,-1.495024e-02,-1.812192e-03,8.774936e-03,4.883809e-05,1.967412e-03,-3.315338e-01,-2.758742e-01,-9.310089e-02,-2.660463e-01
min,0.000000e+00,0.000000e+00,2022-01-01 00:00:00,1.000000e+00,2.022000e+03,-4.191219e+00,-1.996051e+00,-2.519273e+00,-9.970427e-02,-5.614845e-02,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00
25%,1.358000e+03,1.358000e+03,2022-08-06 17:06:00,1.000000e+00,2.022000e+03,-7.493464e-01,-6.917960e-01,-7.628922e-01,-9.970427e-02,-5.614845e-02,-9.362641e-01,-8.029042e-01,-8.660254e-01,-8.660254e-01
50%,1.447000e+03,1.448000e+03,2023-04-13 19:23:00,1.000000e+00,2.023000e+03,1.895027e-01,-8.000990e-02,1.485145e-02,-9.970427e-02,-5.614845e-02,-6.025698e-01,-3.916012e-01,-2.449294e-16,-5.000000e-01
75%,1.565000e+03,1.565000e+03,2023-09-10 20:57:00,1.000000e+00,2.023000e+03,7.707278e-01,6.198615e-01,7.945084e-01,-9.970427e-02,-5.614845e-02,2.005080e-01,1.192704e-01,5.000000e-01,5.000000e-01
max,1.911000e+03,1.911000e+03,2024-06-01 00:00:00,9.000000e+00,2.024000e+03,2.175728e+00,5.576328e+00,2.359562e+00,6.945283e+01,7.287764e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
std,2.191965e+02,2.176947e+02,NaN,2.348621e-01,6.859159e-01,1.003718e+00,1.000607e+00,9.993908e-01,9.976954e-01,1.017557e+00,6.781692e-01,5.950339e-01,7.072976e-01,6.482914e-01


In [16]:
df_agg.dtypes

start_station_idx             int64
end_station_idx               int64
started_minute       datetime64[ns]
n_viajes                      int64
year                          int64
temp_std                    float64
wind_std                    float64
rel_humidity_std            float64
precipitation_std           float64
snow_depth_std              float64
hour_sin                    float64
hour_cos                    float64
month_sin                   float64
month_cos                   float64
event                          bool
normal_day                     bool
weekend_day                    bool
holiday_day                    bool
dtype: object

In [17]:
import pandas as pd
import statsmodels.api as sm

# Selecciona features
X = df_agg[[
    "temp_std",
    "wind_std",
    "rel_humidity_std",
    "precipitation_std",
    "snow_depth_std",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "normal_day",
    "weekend_day",
    "holiday_day",
    "event"
]]

# Convertir booleanos a enteros
bool_cols = ["normal_day", "weekend_day", "holiday_day", "event"]
X[bool_cols] = X[bool_cols].astype(int)

# Variable objetivo
y = df_agg["n_viajes"]

# Añadir constante para statsmodels
X = sm.add_constant(X)

# Entrenar GLM Poisson
poisson_model = sm.GLM(y, X, family=sm.families.Poisson())
poisson_results = poisson_model.fit()

print(poisson_results.summary())


C:\Users\burvu\AppData\Local\Temp\ipykernel_4780\1737302503.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[bool_cols] = X[bool_cols].astype(int)


                 Generalized Linear Model Regression Results                  
Dep. Variable:               n_viajes   No. Observations:              9073410
Model:                            GLM   Df Residuals:                  9073396
Model Family:                 Poisson   Df Model:                           13
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -9.3788e+06
Date:                Thu, 27 Nov 2025   Deviance:                   3.5140e+05
Time:                        14:00:29   Pearson chi2:                 4.63e+05
No. Iterations:                     4   Pseudo R-squ. (CS):          0.0007890
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.6600      0.14

🔹 Lo que obtendrás:

coef → coeficiente del modelo (log-scale)

multiplicador_viajes → cuánto se multiplica el número esperado de viajes por unidad de esa variable

Ordenadas de mayor a menor impacto (en valor absoluto del coef)

In [18]:
import pandas as pd
import numpy as np

# Extraer coeficientes
coefs = poisson_results.params

# Convertir a multiplicadores de viajes
multiplicadores = np.exp(coefs)

# Crear DataFrame
impacto = pd.DataFrame({
    'variable': coefs.index,
    'coef': coefs.values,
    'multiplicador_viajes': multiplicadores
})

# Ordenar por efecto absoluto (mayor impacto)
impacto['abs_coef'] = impacto['coef'].abs()
impacto_sorted = impacto.sort_values(by='abs_coef', ascending=False).drop(columns='abs_coef')

print(impacto_sorted)


                            variable      coef  multiplicador_viajes
weekend_day              weekend_day  0.733851              2.083086
holiday_day              holiday_day  0.720126              2.054692
normal_day                normal_day  0.687620              1.988977
const                          const -0.660021              0.516840
hour_sin                    hour_sin -0.011171              0.988891
month_cos                  month_cos -0.008559              0.991477
temp_std                    temp_std  0.006736              1.006758
event                          event  0.005000              1.005013
hour_cos                    hour_cos  0.004808              1.004819
month_sin                  month_sin -0.003386              0.996620
rel_humidity_std    rel_humidity_std -0.001476              0.998525
snow_depth_std        snow_depth_std -0.000415              0.999585
precipitation_std  precipitation_std -0.000379              0.999621
wind_std                    wind_s